# Aquarius-style streaming reservoir inflow benchmark

This notebook replays the provided Aquarius exports as an online stream, runs Kalmone's reservoir estimator, and compares it with two transparent baselines:

1. the raw storage water-balance inflow; and
2. a strictly causal rolling mean of that raw inflow.

The notebook performs the input cleaning required by Kalmone inside the notebook: Aquarius metadata is skipped, timestamps are parsed as timezone-aware UTC, values are numeric, observations are sorted and deduplicated, component series are causally aligned, and the final storage/outflow indexes are identical, strictly increasing, and ready for the package. Missing values remain `NaN` after initialization because the online API explicitly supports partial observations.

The primary example is Lexington. Its outlet discharge and spillway flow are combined into measured reservoir outflow. Chesbro is supported as a repeatable comparison, but its downstream discharge location makes automatic spillway addition unsafe; the notebook therefore keeps Chesbro spillway flow diagnostic-only unless the hydrologic interpretation is confirmed.

## Evaluation rules

- Replay is event-time based: no future observation is used to align an input.
- The stream uses a backward as-of match with a finite tolerance for off-grid Aquarius readings.
- Raw inflow uses the actual elapsed seconds between timestamps, not a hard-coded 15-minute interval.
- The rolling baseline is trailing and causal.
- Kalmone inflow is evaluated as a causal filtered output; Kalmone outflow is evaluated only after its fixed-lag release.
- Upstream flow is a partial-catchment proxy, not independent total-inflow truth.
- The one-step storage metric asks: given the estimate at time *t*, how well does it predict storage at the next observed timestamp?

In [ ]:
from datetime import timedelta
from pathlib import Path
import os
import sys
import time

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from IPython.display import display

def find_project_root() -> Path:
    explicit_root = os.environ.get("KALMONE_NOTEBOOK_ROOT")
    candidates = ([Path(explicit_root)] if explicit_root else []) + [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "Reservoirs").is_dir() and (candidate / "src" / "kalmone").is_dir():
            return candidate
    raise FileNotFoundError(
        "Could not find the Working project. Launch from the project or set KALMONE_NOTEBOOK_ROOT."
    )

ROOT = find_project_root()
SOURCE_ROOT = ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

from kalmone import (
    OnlineFixedLagRTS,
    OnlineInflowPipeline,
    OnlineReservoirInflow,
    ReservoirBackend,
    ReservoirStateSpaceModel,
    UnitSystem,
)

DATA_ROOT = ROOT / "Reservoirs"
RESERVOIR = "Lexington"  # Change to "Chesbro" to repeat the analysis.
REPLAY_START = pd.Timestamp("2023-12-01", tz="UTC")
REPLAY_END = pd.Timestamp("2024-02-01", tz="UTC")
ASOF_TOLERANCE = pd.Timedelta("20min")
ROLLING_WINDOW = "4h"
OVERVIEW_FREQUENCY = "1D"  # Used only by the input/output context chart.
SMOOTHING_LAG = timedelta(hours=4)
UNITS = UnitSystem.us_customary()

# These are illustrative starting values. They must be re-tuned and independently
# justified before field deployment. Tuning, if used, must be restricted to a
# training period and its result must be frozen before evaluation.
Q_STORAGE = 0.02
Q_INFLOW = 0.30
Q_OUTFLOW = 0.05
R_STORAGE = 20.0**2
R_OUTFLOW = 10.0**2

SOURCE_FILES = {
    "Lexington": {
        "storage": DATA_ROOT / "Lexington/Total_Storage.csv",
        "outlet": DATA_ROOT / "Lexington/Discharge.csv",
        "spillway": DATA_ROOT / "Lexington/Spillway_Flow.csv",
        "upstream": DATA_ROOT / "Lexington/Upstream.csv",
        "combine_spillway": True,
    },
    "Chesbro": {
        "storage": DATA_ROOT / "Chesbro/Total_Storage.csv",
        "outlet": DATA_ROOT / "Chesbro/Discharge.csv",
        "spillway": DATA_ROOT / "Chesbro/Spillway_Flow.csv",
        "upstream": DATA_ROOT / "Chesbro/Upstream.csv",
        "combine_spillway": False,
    },
}
cfg = SOURCE_FILES[RESERVOIR]
print(f"Reservoir: {RESERVOIR}")
print(f"Replay: {REPLAY_START} through {REPLAY_END}")
print(f"Data root: {DATA_ROOT}")

## 1. Aquarius parsing and audit helpers

Aquarius exports contain comment metadata before the data header. The parser keeps the source metadata useful for audit, while the returned series has the exact timezone-aware, numeric shape expected by Kalmone. Duplicate timestamps are resolved deterministically by keeping the last source record and are counted in the audit table.

In [ ]:
def read_aquarius_series(path: Path, name: str) -> tuple[pd.Series, dict[str, object]]:
    """Read one Aquarius CSV and return a cleaned UTC series plus audit metadata."""
    metadata_lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    metadata = {}
    for line in metadata_lines:
        if not line.startswith("#") or ":" not in line:
            continue
        key, value = line[1:].split(":", 1)
        metadata[key.strip()] = value.strip()

    raw = pd.read_csv(path, comment="#")
    required = {"ISO 8601 UTC", "Value"}
    missing_columns = required.difference(raw.columns)
    if missing_columns:
        raise ValueError(f"{path} is missing columns: {sorted(missing_columns)}")

    timestamp = pd.to_datetime(raw["ISO 8601 UTC"], utc=True, errors="coerce")
    values = pd.to_numeric(raw["Value"], errors="coerce")
    parsed = pd.DataFrame({"timestamp": timestamp, name: values})
    parse_failures = int(parsed["timestamp"].isna().sum())
    parsed = parsed.dropna(subset=["timestamp"])
    parsed = parsed.sort_values("timestamp", kind="stable")
    duplicate_count = int(parsed["timestamp"].duplicated(keep="last").sum())
    parsed = parsed.drop_duplicates("timestamp", keep="last")
    parsed = parsed.set_index("timestamp").sort_index()
    if parsed.index.tz is None:
        raise ValueError(f"{path} did not produce a timezone-aware index")

    dt = parsed.index.to_series().diff().dt.total_seconds()
    audit = {
        "name": name,
        "path": str(path),
        "source_rows": len(raw),
        "clean_rows": len(parsed),
        "timestamp_parse_failures": parse_failures,
        "duplicates_removed": duplicate_count,
        "finite_value_fraction": float(parsed[name].notna().mean()),
        "first_utc": parsed.index.min(),
        "last_utc": parsed.index.max(),
        "median_interval_min": float(dt.median() / 60.0),
        "gaps_over_20min": int((dt > 20 * 60).sum()),
        "aquarius_identifier": metadata.get("Time-series identifier", ""),
        "aquarius_location": metadata.get("Location", ""),
        "aquarius_units": metadata.get("Value units", ""),
    }
    return parsed[name].rename(name), audit

def audit_sources(config: dict[str, object]) -> tuple[dict[str, pd.Series], pd.DataFrame]:
    series = {}
    audits = []
    for key in ("storage", "outlet", "spillway", "upstream"):
        value, audit = read_aquarius_series(config[key], key)
        series[key] = value
        audits.append(audit)
    return series, pd.DataFrame(audits).set_index("name")

series, source_audit = audit_sources(cfg)
display(source_audit[["path", "source_rows", "clean_rows", "duplicates_removed", "finite_value_fraction", "first_utc", "last_utc", "median_interval_min", "gaps_over_20min"]])

## 2. Causal alignment and package-contract validation

Storage timestamps define the stream clock. Outlet, spillway, and upstream values are matched using only the most recent source value at or before each storage timestamp. The tolerance prevents a stale sensor value from being treated as current. The final frame retains missing values for Kalmone's documented partial-observation behavior. The stream is trimmed only before its first joint finite storage/outflow observation so initialization cannot fail because of a leading missing discharge.

In [ ]:
def backward_asof_to_clock(clock: pd.DatetimeIndex, source: pd.Series, name: str) -> pd.Series:
    left = pd.DataFrame({"timestamp": clock})
    right = source.rename(name).rename_axis("timestamp").reset_index()
    right = right.sort_values("timestamp", kind="stable")
    aligned = pd.merge_asof(
        left.sort_values("timestamp"),
        right,
        on="timestamp",
        direction="backward",
        tolerance=ASOF_TOLERANCE,
    )
    return aligned.set_index("timestamp")[name].rename(name)

def clean_and_align(series: dict[str, pd.Series], config: dict[str, object]) -> tuple[pd.DataFrame, dict[str, object]]:
    storage = series["storage"].sort_index()
    clock = storage.index
    outlet = backward_asof_to_clock(clock, series["outlet"], "outlet_discharge")
    spillway = backward_asof_to_clock(clock, series["spillway"], "spillway_flow")
    upstream = backward_asof_to_clock(clock, series["upstream"], "upstream_flow")

    aligned = pd.concat(
        [storage.rename("storage"), outlet, spillway, upstream],
        axis=1,
    ).sort_index()
    aligned = aligned[~aligned.index.duplicated(keep="last")]
    if bool(config["combine_spillway"]):
        aligned["measured_outflow"] = aligned["outlet_discharge"] + aligned["spillway_flow"]
        outflow_definition = "outlet discharge + spillway flow"
    else:
        aligned["measured_outflow"] = aligned["outlet_discharge"]
        outflow_definition = "downstream discharge only; spillway retained as diagnostic"

    aligned = aligned.loc[REPLAY_START:REPLAY_END].copy()
    aligned = aligned.replace([np.inf, -np.inf], np.nan)
    first_joint = aligned[["storage", "measured_outflow"]].dropna().index.min()
    if pd.isna(first_joint):
        raise ValueError("No joint finite storage/outflow observation exists in replay window")
    aligned = aligned.loc[first_joint:].copy()

    # These are the exact checks required before passing data to Kalmone.
    if aligned.index.tz is None:
        raise AssertionError("input index must be timezone-aware")
    if not aligned.index.is_monotonic_increasing or aligned.index.has_duplicates:
        raise AssertionError("input index must be sorted and duplicate-free")
    if not aligned.index.to_series().diff().dropna().gt(pd.Timedelta(0)).all():
        raise AssertionError("input timestamps must be strictly increasing")
    storage_input = aligned["storage"]
    outflow_input = aligned["measured_outflow"]
    if not storage_input.index.equals(outflow_input.index):
        raise AssertionError("storage and outflow indexes must match exactly")

    dt = aligned.index.to_series().diff().dt.total_seconds()
    audit = {
        "rows": len(aligned),
        "start": aligned.index.min(),
        "end": aligned.index.max(),
        "storage_missing": int(aligned.storage.isna().sum()),
        "outflow_missing": int(aligned.measured_outflow.isna().sum()),
        "upstream_missing": int(aligned.upstream_flow.isna().sum()),
        "median_interval_min": float(dt.median() / 60),
        "intervals_over_20min": int((dt > 20 * 60).sum()),
        "outflow_definition": outflow_definition,
    }
    return aligned, audit

observations, clean_audit = clean_and_align(series, cfg)
display(pd.DataFrame([clean_audit]))
display(observations[["storage", "outlet_discharge", "spillway_flow", "measured_outflow", "upstream_flow"]].head())
display(observations.isna().mean().rename("missing_fraction").to_frame())

## 3. Raw and rolling baselines

The raw baseline differentiates storage and adds measured outflow. Because Aquarius has gaps and nonuniform timestamps, its interval length is calculated from the index. The rolling baseline is a trailing time-based mean, so it does not use future values.

In [ ]:
dt_seconds = observations.index.to_series().diff().dt.total_seconds()
storage_change = observations["storage"].diff()
raw_inflow = (
    storage_change / (UNITS.flow_to_volume_per_second * dt_seconds)
    + observations["measured_outflow"]
).rename("raw_inflow")
rolling_inflow = raw_inflow.rolling(ROLLING_WINDOW, min_periods=4).mean().rename("rolling_inflow")

observations = observations.assign(
    raw_inflow=raw_inflow,
    rolling_inflow=rolling_inflow,
)
display(observations[["storage", "measured_outflow", "raw_inflow", "rolling_inflow"]].head(12))

## 4. Replay the cleaned stream through Kalmone

Every row below is passed to `OnlineReservoirInflow.process`. A parallel lower-level `OnlineInflowPipeline` exposes the corresponding fixed-lag RTS-smoothed inflow for comparison. Causal inflow is available immediately; the RTS inflow and outflow estimates are released only after the configured smoothing lag, so they are useful delayed diagnostics rather than real-time estimates.

In [ ]:
stream = OnlineReservoirInflow(
    q_storage=Q_STORAGE,
    q_inflow=Q_INFLOW,
    q_outflow=Q_OUTFLOW,
    r_storage=R_STORAGE,
    r_outflow=R_OUTFLOW,
    smoothing_lag=SMOOTHING_LAG,
)

# The public reservoir API exposes causal inflow and finalized outflow.
# This matching lower-level pipeline exposes the fixed-lag RTS inflow state
# for a like-for-like causal-versus-smoothed inflow comparison.
rts_diagnostic_pipeline = OnlineInflowPipeline(
    backend=ReservoirBackend(
        model=ReservoirStateSpaceModel(
            q_continuous=np.diag([Q_STORAGE, Q_INFLOW, Q_OUTFLOW])
        ),
        initial_covariance=np.diag([100.0, 1000.0, 1000.0]),
        observation_covariance=np.diag([R_STORAGE, R_OUTFLOW]),
    ),
    smoother=OnlineFixedLagRTS(SMOOTHING_LAG),
)

inflow_records = []
rts_inflow_records = []
outflow_records = []
process_seconds = []
for timestamp, row in observations.iterrows():
    storage = float(row["storage"]) if pd.notna(row["storage"]) else np.nan
    discharge = (
        float(row["measured_outflow"])
        if pd.notna(row["measured_outflow"])
        else np.nan
    )
    started = time.perf_counter()
    update = stream.process(
        timestamp=timestamp.to_pydatetime(), storage=storage, discharge=discharge
    )
    process_seconds.append(time.perf_counter() - started)
    rts_update = rts_diagnostic_pipeline.process(
        timestamp=timestamp.to_pydatetime(), storage=storage, discharge=discharge
    )
    inflow_records.extend(
        {
            "timestamp": estimate.timestamp,
            "kalmone_inflow": estimate.value,
            "inflow_prediction_flag": str(estimate.prediction_flag),
            "inflow_smoothing_flag": str(estimate.smoothing_flag),
        }
        for estimate in update.filtered_inflows
    )
    rts_inflow_records.extend(
        {
            "timestamp": state.timestamp,
            "kalmone_rts_inflow": float(state.mean[1]),
            "rts_inflow_release_timestamp": timestamp,
            "rts_inflow_prediction_flag": str(state.prediction_flag),
            "rts_inflow_smoothing_flag": "SMOOTHED",
        }
        for state in rts_update.smoothed_states
    )
    outflow_records.extend(
        {
            "timestamp": estimate.timestamp,
            "kalmone_outflow": estimate.value,
            "outflow_release_timestamp": timestamp,
            "outflow_prediction_flag": str(estimate.prediction_flag),
            "outflow_smoothing_flag": str(estimate.smoothing_flag),
        }
        for estimate in update.estimated_outflows
    )

inflows = pd.DataFrame(inflow_records)
if not inflows.empty:
    inflows = inflows.set_index("timestamp").sort_index()
rts_inflows = pd.DataFrame(rts_inflow_records)
if not rts_inflows.empty:
    rts_inflows = rts_inflows.set_index("timestamp").sort_index()
outflows = pd.DataFrame(outflow_records)
if not outflows.empty:
    outflows = outflows.set_index("timestamp").sort_index()

print(f"Input observations: {len(observations):,}")
print(f"Causal inflow outputs: {len(inflows):,}")
print(f"Fixed-lag RTS inflow outputs: {len(rts_inflows):,}")
print(f"Finalized outflow outputs: {len(outflows):,}")
print(f"Pending smoothing states: {stream.pending_count:,}")
print(f"Median public-stream process call: {np.median(process_seconds) * 1e3:.3f} ms")

In [ ]:
# Align stream outputs back to the cleaned input clock. Missing output rows
# are expected during initialization and in the unfinished trailing lag.
comparison = observations.join(inflows[["kalmone_inflow", "inflow_prediction_flag", "inflow_smoothing_flag"]], how="left")
comparison = comparison.join(
    rts_inflows[[
        "kalmone_rts_inflow", "rts_inflow_prediction_flag",
        "rts_inflow_smoothing_flag", "rts_inflow_release_timestamp",
    ]],
    how="left",
)
comparison = comparison.join(outflows[["kalmone_outflow", "outflow_prediction_flag", "outflow_smoothing_flag", "outflow_release_timestamp"]], how="left")

if not outflows.empty:
    release_delay = (
        pd.to_datetime(outflows["outflow_release_timestamp"], utc=True)
        - pd.to_datetime(outflows.index, utc=True)
    ).dt.total_seconds().to_numpy()
    print(f"Fixed-lag release delay: median={np.median(release_delay) / 3600:.2f} h, unique={np.unique(release_delay)} seconds")
    assert np.allclose(release_delay, SMOOTHING_LAG.total_seconds())

if not rts_inflows.empty:
    rts_inflow_delay = (
        pd.to_datetime(rts_inflows["rts_inflow_release_timestamp"], utc=True)
        - pd.to_datetime(rts_inflows.index, utc=True)
    ).dt.total_seconds().to_numpy()
    assert np.allclose(rts_inflow_delay, SMOOTHING_LAG.total_seconds())

display(
    comparison[[
        "raw_inflow", "rolling_inflow", "kalmone_inflow",
        "kalmone_rts_inflow", "kalmone_outflow",
    ]].head(12)
)

## 5. Metrics, including one-step storage prediction

There is no direct total-inflow truth in these exports. The primary comparison therefore reports coverage, stability, causal availability, upstream-proxy agreement, and storage closure rather than treating the upstream gage as ground truth. The fixed-lag RTS inflow is included as a delayed diagnostic; it must not be treated as a same-time causal estimate.

For each inflow estimate at time *t*, the one-step prediction is:

`storage_hat[t+1] = storage[t] + (inflow[t] - measured_outflow[t]) * dt[t] / 43560`

The one-step metric compares that prediction with the next observed storage. It reports RMSE, MAE, bias, and coverage in acre-feet.

In [ ]:
def flow_metrics(estimate: pd.Series, upstream: pd.Series) -> dict[str, float]:
    values = estimate.to_numpy(dtype=float)
    finite = np.isfinite(values)
    result = {
        "coverage": float(finite.mean()),
        "negative_rate": float(np.mean(values[finite] < 0)) if finite.any() else np.nan,
        "estimate_std_cfs": float(np.std(values[finite])) if finite.any() else np.nan,
        "p95_abs_step_cfs": float(np.nanpercentile(np.abs(np.diff(values[finite])), 95)) if finite.sum() > 1 else np.nan,
    }
    proxy_mask = finite & np.isfinite(upstream.to_numpy(dtype=float))
    if proxy_mask.sum() >= 2:
        result["upstream_proxy_points"] = float(proxy_mask.sum())
        result["upstream_proxy_corr"] = float(np.corrcoef(values[proxy_mask], upstream.to_numpy(dtype=float)[proxy_mask])[0, 1])
    else:
        result["upstream_proxy_points"] = 0.0
        result["upstream_proxy_corr"] = np.nan
    return result

def one_step_storage_metrics(storage: pd.Series, outflow: pd.Series, inflow: pd.Series) -> dict[str, float]:
    next_timestamp = pd.Series(storage.index, index=storage.index).shift(-1)
    dt = (next_timestamp - pd.Series(storage.index, index=storage.index)).dt.total_seconds()
    predicted = storage + (inflow - outflow) * UNITS.flow_to_volume_per_second * dt
    truth = storage.shift(-1)
    error = (predicted - truth).to_numpy(dtype=float)
    finite = np.isfinite(error)
    if not finite.any():
        return {"points": 0.0, "coverage": 0.0, "rmse_acre_ft": np.nan, "mae_acre_ft": np.nan, "bias_acre_ft": np.nan}
    return {
        "points": float(finite.sum()),
        "coverage": float(finite.mean()),
        "rmse_acre_ft": float(np.sqrt(np.mean(error[finite] ** 2))),
        "mae_acre_ft": float(np.mean(np.abs(error[finite]))),
        "bias_acre_ft": float(np.mean(error[finite])),
    }

estimators = {
    "raw water balance": comparison["raw_inflow"],
    "4h causal rolling mean": comparison["rolling_inflow"],
    "Kalmone causal filter": comparison["kalmone_inflow"],
    "Kalmone fixed-lag RTS": comparison["kalmone_rts_inflow"],
}
flow_table = pd.DataFrame({name: flow_metrics(values, comparison["upstream_flow"]) for name, values in estimators.items()}).T
storage_table = pd.DataFrame({name: one_step_storage_metrics(comparison["storage"], comparison["measured_outflow"], values) for name, values in estimators.items()}).T
print("Flow behavior and upstream-proxy metrics")
display(flow_table.round(4))
print("One-step-ahead storage prediction metrics")
display(storage_table.round(3))

In [ ]:
# Additional operational checks: missing-data behavior, flags, and release coverage.
flag_table = pd.DataFrame({
    "inflow_prediction_flag_counts": comparison["inflow_prediction_flag"].value_counts(dropna=False),
    "outflow_prediction_flag_counts": comparison["outflow_prediction_flag"].value_counts(dropna=False),
})
display(flag_table)

if not outflows.empty:
    print(f"Finalized outflow coverage: {len(outflows) / len(comparison):.2%}")
    print(f"Trailing unfinished lag rows: {comparison["kalmone_outflow"].isna().sum():,}")
print(f"Rows with missing storage: {comparison.storage.isna().sum():,}")
print(f"Rows with missing measured outflow: {comparison.measured_outflow.isna().sum():,}")

## 6. Interactive visual comparison

The inflow comparison uses every cleaned timestamp and WebGL rendering. The input/output context view uses daily means to stay responsive across the whole record. The fourteen-day detail also retains native observations. Storage and flow rates are always placed on separate y-axes and panels; they are not overlaid because their units differ.

In [ ]:
# Inflow comparison uses every cleaned timestamp. The separate input/output
# context plot below remains aggregated for responsive whole-record browsing.
inflow_columns = [
    "raw_inflow", "rolling_inflow", "kalmone_inflow",
    "kalmone_rts_inflow", "upstream_flow",
]
inflow_comparison = comparison[inflow_columns]
overview_columns = [
    "storage", "measured_outflow", "kalmone_outflow", "upstream_flow",
]
overview = comparison[overview_columns].resample(OVERVIEW_FREQUENCY).mean()

def line_trace(
    frame, column, label, color, *, width=1.5, opacity=1.0, dash="solid",
    showlegend=True, legendgroup=None, legendgrouptitle=None, webgl=False,
):
    trace_options = {
        "showlegend": showlegend, "legendgroup": legendgroup,
    }
    if legendgrouptitle is not None:
        trace_options["legendgrouptitle"] = {"text": legendgrouptitle}
    trace_type = go.Scattergl if webgl else go.Scatter
    return trace_type(
        x=frame.index, y=frame[column], mode="lines", name=label,
        connectgaps=False, opacity=opacity,
        line={"color": color, "width": width, "dash": dash},
        hovertemplate="%{y:,.2f}<extra>" + label + "</extra>",
        **trace_options,
    )

fig = go.Figure()
for column, label, color, width, opacity, dash in [
    ("raw_inflow", "Raw water balance", "#d62728", 0.8, 0.38, "solid"),
    ("rolling_inflow", f"{ROLLING_WINDOW} causal rolling mean", "#ff7f0e", 1.4, 0.9, "solid"),
    ("kalmone_inflow", "Kalmone causal inflow", "#1f77b4", 2.0, 1.0, "solid"),
    ("kalmone_rts_inflow", f"Kalmone fixed-lag RTS ({SMOOTHING_LAG})", "#17becf", 1.8, 1.0, "dash"),
    ("upstream_flow", "Upstream proxy", "#9467bd", 1.2, 0.8, "solid"),
]:
    fig.add_trace(
        line_trace(
            inflow_comparison, column, label, color, width=width, opacity=opacity,
            dash=dash, webgl=True,
        )
    )
fig.update_layout(
    title=f"{RESERVOIR}: every-step inflow comparison",
    xaxis={"title": "UTC timestamp", "rangeslider": {"visible": True}},
    yaxis_title="Flow (cfs)", hovermode="x unified",
    template="plotly_white", height=560,
    margin={"l": 75, "r": 260, "t": 80, "b": 55},
    legend={"orientation": "v", "x": 1.02, "xanchor": "left", "y": 1.0, "yanchor": "top"},
)
fig.show()

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.09,
    subplot_titles=("Storage (daily mean)", "Outflow and upstream proxy (daily mean)"),
)
fig.add_trace(line_trace(overview, "storage", "Storage", "#7f7f7f", width=1.8), row=1, col=1)
fig.add_trace(line_trace(overview, "measured_outflow", "Measured outflow", "#ff7f0e", width=1.4), row=2, col=1)
fig.add_trace(line_trace(overview, "kalmone_outflow", "Kalmone fixed-lag outflow", "#2ca02c", width=1.8), row=2, col=1)
fig.add_trace(line_trace(overview, "upstream_flow", "Upstream proxy", "#9467bd", width=1.2, opacity=0.8), row=2, col=1)
fig.update_yaxes(title_text="Storage (acre-ft)", row=1, col=1)
fig.update_yaxes(title_text="Flow (cfs)", row=2, col=1)
fig.update_xaxes(title_text="UTC timestamp", rangeslider={"visible": True}, row=2, col=1)
fig.update_layout(
    title="Inputs and fixed-lag output", hovermode="x unified",
    template="plotly_white", height=680,
    legend={"orientation": "h", "yanchor": "bottom", "y": 1.02, "x": 0},
)
fig.show()

In [ ]:
# This native-resolution detail is intentionally not aggregated.
excerpt_start = comparison.index[-1] - pd.Timedelta(days=14)
excerpt = comparison.loc[excerpt_start:]
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    row_heights=[0.42, 0.27, 0.31],
    subplot_titles=("Inflow detail", "Storage", "Outflow and upstream proxy"),
)
for column, label, color, width, opacity, dash in [
    ("raw_inflow", "Raw water balance", "#d62728", 1.0, 0.6, "solid"),
    ("rolling_inflow", f"{ROLLING_WINDOW} rolling mean", "#ff7f0e", 1.5, 1.0, "solid"),
    ("kalmone_inflow", "Kalmone causal inflow", "#1f77b4", 2.0, 1.0, "solid"),
    ("kalmone_rts_inflow", f"Kalmone fixed-lag RTS ({SMOOTHING_LAG})", "#17becf", 1.8, 1.0, "dash"),
    ("upstream_flow", "Upstream proxy", "#9467bd", 1.2, 0.8, "solid"),
]:
    fig.add_trace(
        line_trace(
            excerpt, column, label, color, width=width, opacity=opacity, dash=dash,
            legendgroup="inflow", legendgrouptitle="Inflow",
        ),
        row=1, col=1,
    )
fig.add_trace(
    line_trace(excerpt, "storage", "Storage", "#7f7f7f", width=1.8, showlegend=False),
    row=2, col=1,
)
fig.add_trace(
    line_trace(
        excerpt, "measured_outflow", "Measured outflow", "#ff7f0e", width=1.4,
        legendgroup="outflow", legendgrouptitle="Outflow",
    ),
    row=3, col=1,
)
fig.add_trace(
    line_trace(
        excerpt, "kalmone_outflow", "Kalmone fixed-lag", "#2ca02c", width=1.8,
        legendgroup="outflow",
    ),
    row=3, col=1,
)
fig.add_trace(
    line_trace(
        excerpt, "upstream_flow", "Upstream proxy", "#9467bd", width=1.2, opacity=0.8,
        showlegend=False, legendgroup="inflow",
    ),
    row=3, col=1,
)
fig.add_hline(y=0, line_width=1, line_color="gray", row=1, col=1)
fig.update_yaxes(title_text="Inflow (cfs)", row=1, col=1)
fig.update_yaxes(title_text="Storage (acre-ft)", row=2, col=1)
fig.update_yaxes(title_text="Flow (cfs)", row=3, col=1)
fig.update_xaxes(title_text="UTC timestamp", rangeslider={"visible": True}, row=3, col=1)
fig.update_layout(
    title="Interactive fourteen-day detail", hovermode="x unified",
    template="plotly_white", height=900,
    margin={"l": 75, "r": 220, "t": 80, "b": 55},
    legend={
        "orientation": "v", "x": 1.02, "xanchor": "left",
        "y": 1.0, "yanchor": "top", "tracegroupgap": 12,
    },
)
fig.show()

## Interpretation and limitations

The raw water-balance series is an accounting calculation and will expose storage differencing noise. The rolling mean is a useful low-complexity smoothing comparator but has a controllable response delay. Kalmone provides a causal filtered inflow plus fixed-lag RTS-smoothed inflow and outflow diagnostics; the smoothed products are delayed by the configured lag.

The one-step storage table is the most direct internal consistency check available without an independent inflow sensor. Upstream agreement is only a proxy because the upstream gage does not measure every tributary or reservoir inflow component. Before operational use, replace the illustrative noise parameters with a documented tuning exercise on a training period, evaluate on a held-out period, and confirm the outlet/spillway accounting for each reservoir.